# Notebook 01: Load Dataset & Labels

**Objective:**
Parse the `quality-hr-ann.csv` and `subject-info.csv` from the BUT PPG dataset and create a standardized `labels.csv` mapping.

In [1]:
import sys
import os
import pandas as pd

project_root = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import BUTPPGDataLoader

DATA_DIR = os.path.join(project_root, 'brno-university-of-technology-smartphone-ppg-database-but-ppg-2.0.0')
PROCESSED_DIR = os.path.join(project_root, 'processed_data')
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [2]:
loader = BUTPPGDataLoader(DATA_DIR)
record_ids = loader.get_all_record_ids()
print(f"Total records in dataset: {len(record_ids)}")

Total records in dataset: 3888


In [3]:
labels_data = []
for rid in record_ids:
    meta = loader.get_metadata(rid)
    quality = meta.get('Quality', -1)
    motion = str(meta.get('Motion', '0'))
    
    # Logic: 3-Class Definition
    if quality == 1:
        label = 'Poor'
    elif quality == 0 and motion != '0':
        label = 'Moderate'
    elif quality == 0 and motion == '0':
        label = 'Good'
    else:
        continue
        
    labels_data.append({'ID': rid, 'Quality': quality, 'Motion': motion, 'SQI_Label': label})

df_labels = pd.DataFrame(labels_data)
df_labels.to_csv(os.path.join(PROCESSED_DIR, 'labels_3class.csv'), index=False)
print("Saved labels_3class.csv")

Saved labels_3class.csv


### Validation Check (Crucial)

In [4]:
print("Data Shape:", df_labels.shape)
print("Columns:", df_labels.columns.tolist())
print("\nHead:")
print(df_labels.head())

Data Shape: (3888, 4)
Columns: ['ID', 'Quality', 'Motion', 'SQI_Label']

Head:
       ID  Quality Motion SQI_Label
0  100001        1      0      Poor
1  100002        1      0      Poor
2  100003        1      0      Poor
3  100004        0    2;3  Moderate
4  101001        1      0      Poor
